# JetRacer - Tune bam vach + Lai tay + Thu data train

Mot notebook lam ba viec:

1. **Xem camera & chinh nguong** bam vach dut o giua.
2. **Lai bang tay cam** de thu data train model (CV van chay song song de so sanh).
3. **Cho CV tu chay** de xem no da on chua.

**Truoc moi lan Run All**: `Kernel > Restart & Clear Output`. Hai notebook mo camera cung luc se bao `Failed to create CaptureSession`. Restart cung la cach duy nhat de code moi copy len xe co hieu luc (Python cache module da import).

**AN TOAN**

- Mo len la trang thai DUNG. Khong lenh nao xuong phan cung cho den khi bam **LAI TAY** hoac **CHAY**.
- **LAI TAY** va **CHAY** loai tru nhau - khong bao gio chay cung luc.
- Vao LAI TAY bi tu choi neu can gat chua ve giua.
- Vao CHAY bi tu choi neu dang mat vach tren 20% frame.
- **DUNG KHAN CAP** cat ga ngay. Camera dung hoac loi -> tu dong cat ga.
- Lan dau: **ke banh khoi mat dat**.

## 1. Kiem tra thu muc va file

In [ ]:
%cd /home/jetson/JetsonRacer

import os
import socket

# ===================== CHON NGUON BAM VACH =====================
# True  -> model CNN chay TensorRT (can models/lane_tiny.engine da build tren XE)
# False -> CV co dien nhu truoc
DUNG_CNN = True
# ===============================================================

OVERRIDES = ['configs/cnn.yaml'] if DUNG_CNN else []

required = [
    'tools/tune_lane_jupyter.py',
    'src/jetracer_baseline/tuning_ui.py',
    'src/jetracer_baseline/perception/lane.py',
    'src/jetracer_baseline/perception/shading.py',
    'configs/default.yaml',
]
if DUNG_CNN:
    required += [
        'src/jetracer_baseline/perception/lane_cnn.py',
        'configs/cnn.yaml',
        'models/lane_tiny.engine',
    ]

missing = [p for p in required if not os.path.exists(p)]
print('Hostname:', socket.gethostname())
print('Thu muc:', os.getcwd())
print('Nguon bam vach:', 'CNN (TensorRT)' if DUNG_CNN else 'CV co dien')
print('Files:', 'OK' if not missing else 'THIEU ' + ', '.join(missing))
if missing:
    if 'models/lane_tiny.engine' in missing:
        print()
        print('Chua build engine. Chay trong Terminal (KHONG phai o day):')
        print('  /usr/src/tensorrt/bin/trtexec --onnx=models/lane_tiny.onnx \')
        print('      --fp16 --saveEngine=models/lane_tiny.engine')
        print('Engine phai build TREN CHINH XE - khong chep tu may khac sang.')
    raise IOError('Thieu file, xem danh sach o tren')

print('Shading da hieu chuan:', os.path.exists('configs/shading.yaml'))

if DUNG_CNN:
    # Kiem tra engine nap duoc TRUOC khi mo giao dien. Loi o day de doc hon
    # nhieu so voi loi chon trong luc widget dang dung len.
    import sys
    sys.path.insert(0, 'src')
    from jetracer_baseline.config import load_config
    from jetracer_baseline.perception import build_lane_detector
    _det = build_lane_detector(load_config('configs/default.yaml', OVERRIDES))
    print('Engine nap OK:', type(_det).__name__)
    del _det

## 2. Mo giao dien

Thu tu:

1. **1. MO CAMERA**
2. Tab **Tay cam + Data** -> bam **KET NOI TAY CAM**. Neu bao CHUA KET NOI: **bam/xoay can mot cai** roi bam lai (trinh duyet chi gui su kien gamepad sau thao tac dau tien - day la hanh vi cua Gamepad API, khong phai loi).
3. Kiem tra `Truc lai` / `Truc ga` doc ra dung khi ban gat can. Sai thi doi so truc hoac tick `Dao lai` / `Dao ga`.
4. **KE BANH**, bam **LAI TAY**, thu gat can xem banh quay dung chieu.
5. Chinh mask o tab **Bam vach** cho sach (xem muc 4).
6. Dat xe xuong, bam **GHI DATA (train)** roi lai vai vong.
7. Muon xem CV tu chay: bam **DUNG LAI TAY** truoc, roi bam **CHAY**.

In [ ]:
%run tools/tune_lane_jupyter.py

## 3. Chinh bo goc va toc do khi LAI TAY

Tab **Tay cam + Data** co 8 slider RIENG cho che do lai tay. Chung KHONG anh huong che do tu dong (che do do dung tab "Dieu khien").

| Slider | Dung khi |
|---|---|
| **Bo goc toi da** | Cua khong du gat -> tang. Xem canh bao HARD-LIMIT o duoi |
| **Do mem lai (expo)** | Lai qua nhay quanh vi tri giua -> tang |
| **Toc do doi lai** | Banh lai giat khi gat nhanh -> giam |
| **Toc do toi da** | Xe cham qua -> tang |
| **Ga khoi dong** | Gat nhe ma xe khong lan banh -> tang |
| **Toc do len ga** | Xe vot qua nhanh -> giam |
| **Toc do nha ga** | Nha can ma xe con troi -> tang |
| **Vung chet can** | Tha can ma xe van bo -> tang |

Keo slider an NGAY, khong phai mo lai giao dien, va khong lam khuc lenh dang chay - tune duoc trong luc dang lai.

**Banner khi lai tay hien lenh CUA BAN, khong phai lenh CV.** Vi du `lai+1.00 servo-0.40 HARD-LIMIT` nghia la ban da gat het co nhung servo bi chan - keo `Bo goc toi da` len nua cung vo ich, phai lam muc 6.

## 4. Thu data de train model

Bam **GHI DATA (train)** -> ghi vao `data/driving/<session>_<gio>/`:

```
images/frame_000000.jpg ...
labels.csv
metadata.json
```

| Cot | Y nghia |
|---|---|
| `steering_cmd`, `throttle_cmd` | **Nhan de train** - lenh NGUOI lai |
| `cv_steer`, `cv_throttle` | Lenh CV truyen thong **tren cung frame** |
| `cte`, `curvature`, `drive_mode`, `n_bands`, `lane_found` | Trang thai CV luc do |

Cho nao `steering_cmd` lech xa `cv_steer` la cho CV dang sai - do chinh la cac frame dang gia nhat de train.

Anh la frame **THO chua resize** nhung DA sua mau shading (moi thu ra khoi camera deu sach vien do).

**Thu theo session rieng** (doi `Session:` truoc moi lan): `giua`, `lech_trai`, `lech_phai`, `toi`, `cua_gat`. Chia train/val/test phai chia THEO SESSION.

Muon xem duong di: bam them **GHI VIDEO** -> `logs/tune_<gio>.avi` + `.sidecar.csv`. Gui ca hai ve cho toi.

## 5. Xem thu ma chac chan banh khong quay

Chi can khi muon dat xe tren ban ma van yen tam bam nut. Binh thuong KHONG can chay cell nay.

### Chay voi model CNN

`OVERRIDES` lay tu cong tac `DUNG_CNN` o cell dau. Khi bat:

* Panel **CAMERA + MASK** hien mask do MODEL xuat ra (khong phai nguong mau).
* Panel **BIRD'S-EYE** van ve duong fit nhu cu.
* Cac slider nguong mau (HSV, threshold) **khong con tac dung** - model khong
  dung nguong mau. Slider PID va ga thi van tac dung binh thuong.

Thu tu an toan lan dau: `driver_kind='dryrun'` -> nhin mask va `cte` -> ke banh
khoi mat dat roi doi sang `'nvidia'` -> cuoi cung moi ha xe xuong sa ban.

In [ ]:
try:
    ui.close()
except NameError:
    pass

from tools.tune_lane_jupyter import launch

# driver_kind='dryrun' -> KHONG lenh nao xuong phan cung, chi xem detect.
#              'nvidia' -> dieu khien xe that, NHUNG van chi khi bam nut CHAY.
# Lan dau chay CNN: de 'dryrun', xem mask va cte da dung chua roi moi doi.
ui = launch(driver_kind='dryrun', overrides=OVERRIDES)

## 5. Chay luot day du bang config vua luu

Giao dien tune KHONG ghi log CSV. So lieu chinh thuc phai lay tu mot luot chay day du qua CLI - do moi la con so dua vao Technical Paper.

In [ ]:
!python3 -m src.jetracer_baseline.cli run --task speed --driver nvidia \n    --override configs/tuned.yaml --max-seconds 60 --record

## 6. Dong truoc khi tat notebook

In [ ]:
ui.close()

## 6. Goc cua qua rong - cach be gat toi da

Banner hien `lai-0.60 servo+0.39 TRAN LAI` nghia la: bo dieu khien doi lai 0.60 nhung servo chi quay duoc 0.39 tren thang do 1.0 - tuc **chi dung 39% tam quay** cua servo. Banh khong the be gat hon cho den khi noi gioi han.

Ba gioi han NHAN DON nhau:

```
steer_max 0.60  x  driver.steering_gain 0.65  =  0.390
roi con bi chan boi driver.steering_output_max 0.40
```

**Buoc 1 - DO gioi han co khi that (BAT BUOC, xe KE BANH):**

```
python3 tools/check_hardware.py --driver nvidia --calibrate-steering --wheels-are-lifted
```

Tool quet cham output servo ra tung ben. **Nhin that ky banh truoc**, bam `Ctrl+C` NGAY KHI banh vua cham gam/hoc banh. Dong `output=` in ngay TRUOC dong cuoi la gia tri con an toan.

Khong duoc doan so nay: servo dam vao chan co khi se stall, chay servo hoac sut nguon ca Jetson.

**Buoc 2 - dat vao `configs/default.yaml`:**

```yaml
control:
  steer_max: 1.0
  corner_steer_max: 1.0
  driver:
    steering_gain: -1.0
    steering_output_min: -0.85   # thay bang so DO DUOC o buoc 1
    steering_output_max:  0.85
```

**Buoc 3 - kiem tra:** mo lai giao dien, vao cua thi banner phai hien `servo+0.85` va **khong con** chu `TRAN LAI`.

Neu chi muon cua gat ma doan thang van em: giu `steer_max` thap (0.6) va chi nang `corner_steer_max` len 1.0.